# 09 — Multi-Agent Reinforcement Learning

Most real systems involve multiple interacting agents. MARL introduces fundamental challenges that don't exist in single-agent RL: non-stationarity, credit assignment in cooperative settings, and equilibrium convergence in competitive settings.

## What makes MARL different

**Non-stationarity**: from agent A's perspective, the environment is non-stationary because agent B is also learning. Standard single-agent convergence guarantees break down.

**Cooperative MARL** (shared reward) is better-studied. The dominant paradigm is **CTDE** — Centralised Training, Decentralised Execution: agents share information during training but act on their own observations at test time.

**Competitive MARL**: the goal is a Nash equilibrium — no agent benefits from unilaterally deviating. Finding Nash equilibria is NP-hard in general.

**Self-play** trains an agent against a copy of itself, creating an automatic curriculum. It is the key mechanism behind AlphaGo/Zero, OpenAI Five (Dota 2), and AlphaStar (StarCraft II). The main risk is **cycling** — agent A beats B, B beats C, C beats A — mitigated by league training (maintaining a diverse pool of historical policies).

In [ ]:
# Self-play demo: competitive grid game
import numpy as np
import torch, torch.nn as nn, torch.optim as optim
import matplotlib.pyplot as plt

torch.manual_seed(42); np.random.seed(42)

class CompetitiveGrid:
    def __init__(self, size=5):
        self.size = size; self.goal = (0, 4)

    def reset(self):
        self.pos = [(4, 0), (4, 4)]
        return self._obs()

    def _obs(self):
        return [np.array([*self.pos[i], *self.pos[1-i], *self.goal], dtype=np.float32)/self.size
                for i in range(2)]

    def step(self, actions):
        drs = [(-1,0),(1,0),(0,-1),(0,1),(0,0)]
        for i, a in enumerate(actions):
            r, c = self.pos[i]; dr, dc = drs[a]
            self.pos[i] = (max(0,min(self.size-1,r+dr)), max(0,min(self.size-1,c+dc)))
        rewards = [0.0, 0.0]; done = False
        for i in range(2):
            if self.pos[i] == self.goal:
                rewards[i] = 1.0; rewards[1-i] = -1.0; done = True; break
        return self._obs(), rewards, done

class PolicyNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(6,64),nn.ReLU(),nn.Linear(64,64),nn.ReLU(),nn.Linear(64,5))
    def forward(self, x): return self.net(x)

def select_action(net, obs, epsilon=0.1):
    if np.random.rand() < epsilon: return np.random.randint(5)
    with torch.no_grad(): return net(torch.FloatTensor(obs)).argmax().item()

net = PolicyNet()
optimizer = optim.Adam(net.parameters(), lr=1e-3)
buf = []; epsilon = 1.0; win_rates = []
env = CompetitiveGrid()

for ep in range(3000):
    obs = env.reset()
    for _ in range(50):
        actions = [select_action(net, obs[i], epsilon) for i in range(2)]
        next_obs, rewards, done = env.step(actions)
        for i in range(2):
            buf.append((obs[i], actions[i], rewards[i], next_obs[i], float(done)))
        obs = next_obs
        if done: break
    buf = buf[-5000:]

    if len(buf) >= 64:
        batch = [buf[i] for i in np.random.choice(len(buf), 64, replace=False)]
        s,a,r,ns,dn = zip(*batch)
        s = torch.FloatTensor(np.array(s)); a = torch.LongTensor(a)
        r = torch.FloatTensor(r); ns = torch.FloatTensor(np.array(ns)); dn = torch.FloatTensor(dn)
        q_vals = net(s).gather(1, a.unsqueeze(1)).squeeze(1)
        with torch.no_grad(): target = r + 0.95*net(ns).max(1)[0]*(1-dn)
        loss = nn.functional.mse_loss(q_vals, target)
        optimizer.zero_grad(); loss.backward(); optimizer.step()

    epsilon = max(0.05, epsilon * 0.999)

    if (ep+1) % 200 == 0:
        wins = 0
        for _ in range(100):
            obs_e = env.reset()
            for __ in range(50):
                a0 = select_action(net, obs_e[0], epsilon=0.0)
                obs_e, rwds, d = env.step([a0, np.random.randint(5)])
                if d:
                    if rwds[0] > 0: wins += 1
                    break
        win_rates.append(wins/100)
        print(f"  Ep {ep+1:4d} | Win rate vs random: {wins/100:.1%} | epsilon={epsilon:.3f}")

plt.figure(figsize=(8,4))
plt.plot(range(200,200*(len(win_rates)+1),200), win_rates, 'o-', color='steelblue')
plt.axhline(0.5, color='gray', linestyle='--', alpha=0.5, label='Random baseline')
plt.xlabel('Episode'); plt.ylabel('Win rate vs random')
plt.title('Self-play: Win Rate vs Random Opponent', fontweight='bold')
plt.legend(); plt.tight_layout()
plt.savefig('/tmp/selfplay.png', dpi=100, bbox_inches='tight')
plt.show()
